In [111]:
import xarray as xr
import numpy as np
import torch

# Create a 2x2x2 random DataArray
data = np.random.rand(2, 2, 2)  # Random values between 0 and 1

# Define the dimensions and coordinates
dims = ["x", "y", "z"]
coords = {
    "x": ["A", "B"],  # Labels for the x dimension
    "y": [10, 20],    # Labels for the y dimension
    "z": ["P", "Q"],  # Labels for the z dimension
}

# Create the DataArray
A = xr.DataArray(data, dims=dims, coords=coords)

print(A)

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 64B
array([[[0.99866692, 0.79439934],
        [0.33171298, 0.28711696]],

       [[0.57617926, 0.9067433 ],
        [0.05857525, 0.38869932]]])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'


In [112]:
B = xr.DataArray(np.random.rand(2, 2, 2) , dims=dims, coords=coords).astype(np.float32)
B

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 32B
array([[[0.37131158, 0.58711374],
        [0.48048896, 0.4013294 ]],

       [[0.96914303, 0.69221115],
        [0.47002873, 0.84808904]]], dtype=float32)
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'

In [113]:
A_torch = torch.tensor(A.data, dtype=torch.float32)
B_torch = torch.tensor(B.data, dtype=torch.float32)

In [114]:
A_torch

tensor([[[0.9987, 0.7944],
         [0.3317, 0.2871]],

        [[0.5762, 0.9067],
         [0.0586, 0.3887]]])

In [82]:
torch.set_printoptions(precision=4)

In [97]:
A_bis = A.copy()
print(A_bis)
A_bis[:] = A_torch
print(A_bis)

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 32B
array([[[0.3488853 , 0.89987916],
        [0.8276672 , 0.6674841 ]],

       [[0.5342326 , 0.05152291],
        [0.77237093, 0.65637195]]], dtype=float32)
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'
<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 32B
array([[[0.3488853 , 0.89987916],
        [0.8276672 , 0.6674841 ]],

       [[0.5342326 , 0.05152291],
        [0.77237093, 0.65637195]]], dtype=float32)
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'


In [70]:
def get_density_at_surface(thetao, so, tmask):
    """
    Compute potential density referenced at the surface.

    Parameters:
        thetao (numpy.array) : Temperature array - (t,z,y,x).
        so (numpy.array)     : Salinity array    - (t,z,y,x).
        tmask (numpy.array)  : Mask array        - (t,z,y,x).

    Returns:
        tuple: A tuple containing:
            array: Potential density referenced at the surface.
    """
    rdeltaS = 32.0
    r1_S0 = 0.875 / 35.16504
    r1_T0 = 1.0 / 40.0
    r1_Z0 = 1.0e-4

    EOS000 = 8.0189615746e02
    EOS100 = 8.6672408165e02
    EOS200 = -1.7864682637e03
    EOS300 = 2.0375295546e03
    EOS400 = -1.2849161071e03
    EOS500 = 4.3227585684e02
    EOS600 = -6.0579916612e01
    EOS010 = 2.6010145068e01
    EOS110 = -6.5281885265e01
    EOS210 = 8.1770425108e01
    EOS310 = -5.6888046321e01
    EOS410 = 1.7681814114e01
    EOS510 = -1.9193502195
    EOS020 = -3.7074170417e01
    EOS120 = 6.1548258127e01
    EOS220 = -6.0362551501e01
    EOS320 = 2.9130021253e01
    EOS420 = -5.4723692739
    EOS030 = 2.1661789529e01
    EOS130 = -3.3449108469e01
    EOS230 = 1.9717078466e01
    EOS330 = -3.1742946532
    EOS040 = -8.3627885467
    EOS140 = 1.1311538584e01
    EOS240 = -5.3563304045
    EOS050 = 5.4048723791e-01
    EOS150 = 4.8169980163e-01
    EOS060 = -1.9083568888e-01
    EOS001 = 1.9681925209e01
    EOS101 = -4.2549998214e01
    EOS201 = 5.0774768218e01
    EOS301 = -3.0938076334e01
    EOS401 = 6.6051753097
    EOS011 = -1.3336301113e01
    EOS111 = -4.4870114575
    EOS211 = 5.0042598061
    EOS311 = -6.5399043664e-01
    EOS021 = 6.7080479603
    EOS121 = 3.5063081279
    EOS221 = -1.8795372996
    EOS031 = -2.4649669534
    EOS131 = -5.5077101279e-01
    EOS041 = 5.5927935970e-01
    EOS002 = 2.0660924175
    EOS102 = -4.9527603989
    EOS202 = 2.5019633244
    EOS012 = 2.0564311499
    EOS112 = -2.1311365518e-01
    EOS022 = -1.2419983026
    EOS003 = -2.3342758797e-02
    EOS103 = -1.8507636718e-02
    EOS013 = 3.7969820455e-01
    #set_trace()
    zt = thetao * r1_T0  # temperature
    zs = np.sqrt(np.abs(so + rdeltaS) * r1_S0)  # square root salinity
    print('zt', zt)
    ztm = tmask.squeeze()
    print('zs', zs)

    zn3 = EOS013 * zt + EOS103 * zs + EOS003
    zn2 = (
        (EOS022 * zt + EOS112 * zs + EOS012) * zt + (EOS202 * zs + EOS102) * zs + EOS002
    )
    zn1 = (
        (
            (
                (EOS041 * zt + EOS131 * zs + EOS031) * zt
                + (EOS221 * zs + EOS121) * zs
                + EOS021
            )
            * zt
            + ((EOS311 * zs + EOS211) * zs + EOS111) * zs
            + EOS011
        )
        * zt
        + (((EOS401 * zs + EOS301) * zs + EOS201) * zs + EOS101) * zs
        + EOS001
    )
    zn0 = (
        (
            (
                (
                    (
                        (EOS060 * zt + EOS150 * zs + EOS050) * zt
                        + (EOS240 * zs + EOS140) * zs
                        + EOS040
                    )
                    * zt
                    + ((EOS330 * zs + EOS230) * zs + EOS130) * zs
                    + EOS030
                )
                * zt
                + (((EOS420 * zs + EOS320) * zs + EOS220) * zs + EOS120) * zs
                + EOS020
            )
            * zt
            + ((((EOS510 * zs + EOS410) * zs + EOS310) * zs + EOS210) * zs + EOS110)
            * zs
            + EOS010
        )
        * zt
        + (
            ((((EOS600 * zs + EOS500) * zs + EOS400) * zs + EOS300) * zs + EOS200) * zs
            + EOS100
        )
        * zs
        + EOS000
    )

    rhop = zn0 * ztm  # potential density referenced at the surface
    return rhop, zt

In [71]:
def get_density_at_surface_tensor(thetao, so, tmask):
    """
    Compute potential density referenced at the surface using PyTorch tensors.

    Parameters:
        thetao (torch.Tensor): Temperature tensor - (t, z, y, x).
        so (torch.Tensor): Salinity tensor - (t, z, y, x).
        tmask (torch.Tensor): Mask tensor - (t, z, y, x).

    Returns:
        torch.Tensor: Potential density referenced at the surface.
    """
    # Constants
    rdeltaS = 32.0
    r1_S0 = 0.875 / 35.16504
    r1_T0 = 1.0 / 40.0

    # EOS coefficients
    EOS000 = 8.0189615746e02
    EOS100 = 8.6672408165e02
    EOS200 = -1.7864682637e03
    EOS300 = 2.0375295546e03
    EOS400 = -1.2849161071e03
    EOS500 = 4.3227585684e02
    EOS600 = -6.0579916612e01
    EOS010 = 2.6010145068e01
    EOS110 = -6.5281885265e01
    EOS210 = 8.1770425108e01
    EOS310 = -5.6888046321e01
    EOS410 = 1.7681814114e01
    EOS510 = -1.9193502195
    EOS020 = -3.7074170417e01
    EOS120 = 6.1548258127e01
    EOS220 = -6.0362551501e01
    EOS320 = 2.9130021253e01
    EOS420 = -5.4723692739
    EOS030 = 2.1661789529e01
    EOS130 = -3.3449108469e01
    EOS230 = 1.9717078466e01
    EOS330 = -3.1742946532
    EOS040 = -8.3627885467
    EOS140 = 1.1311538584e01
    EOS240 = -5.3563304045
    EOS050 = 5.4048723791e-01
    EOS150 = 4.8169980163e-01
    EOS060 = -1.9083568888e-01
    EOS001 = 1.9681925209e01
    EOS101 = -4.2549998214e01
    EOS201 = 5.0774768218e01
    EOS301 = -3.0938076334e01
    EOS401 = 6.6051753097
    EOS011 = -1.3336301113e01
    EOS111 = -4.4870114575
    EOS211 = 5.0042598061
    EOS311 = -6.5399043664e-01
    EOS021 = 6.7080479603
    EOS121 = 3.5063081279
    EOS221 = -1.8795372996
    EOS031 = -2.4649669534
    EOS131 = -5.5077101279e-01
    EOS041 = 5.5927935970e-01
    EOS002 = 2.0660924175
    EOS102 = -4.9527603989
    EOS202 = 2.5019633244
    EOS012 = 2.0564311499
    EOS112 = -2.1311365518e-01
    EOS022 = -1.2419983026
    EOS003 = -2.3342758797e-02
    EOS103 = -1.8507636718e-02
    EOS013 = 3.7969820455e-01

    # Tensor computations
    zt = thetao * r1_T0  # temperature
    zs = torch.sqrt(torch.abs(so + rdeltaS) * r1_S0)  # square root salinity
    print('zt', zt)
    ztm = tmask.squeeze()
    print('zs', zs)

    

    zn3 = EOS013 * zt + EOS103 * zs + EOS003
    zn2 = ((EOS022 * zt + EOS112 * zs + EOS012) * zt + (EOS202 * zs + EOS102) * zs + EOS002)
    zn1 = (((((EOS041 * zt + EOS131 * zs + EOS031) * zt + (EOS221 * zs + EOS121) * zs + EOS021) * zt
            + ((EOS311 * zs + EOS211) * zs + EOS111) * zs + EOS011) * zt
            + (((EOS401 * zs + EOS301) * zs + EOS201) * zs + EOS101) * zs + EOS001))
    zn0 = (((((((EOS060 * zt + EOS150 * zs + EOS050) * zt + (EOS240 * zs + EOS140) * zs + EOS040) * zt
              + ((EOS330 * zs + EOS230) * zs + EOS130) * zs + EOS030) * zt
              + (((EOS420 * zs + EOS320) * zs + EOS220) * zs + EOS120) * zs + EOS020) * zt
              + ((((EOS510 * zs + EOS410) * zs + EOS310) * zs + EOS210) * zs + EOS110) * zs + EOS010) * zt
              + ((((EOS600 * zs + EOS500) * zs + EOS400) * zs + EOS300) * zs + EOS200) * zs + EOS100) * zs + EOS000)
    rhop = zn0 * ztm  # potential density referenced at the surface
    return rhop, zt

In [98]:
mask = np.ones((2,2,2))
mask[0,0,0]= 0
mask_xr = xr.DataArray(mask, dims=dims, coords=coords)
mask_xr

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 64B
array([[[0., 1.],
        [1., 1.]],

       [[1., 1.],
        [1., 1.]]])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'

In [99]:
density, zn0 = get_density_at_surface(A,B,mask)

zt <xarray.DataArray (x: 2, y: 2, z: 2)> Size: 32B
array([[[0.00872213, 0.02249698],
        [0.02069168, 0.0166871 ]],

       [[0.01335582, 0.00128807],
        [0.01930927, 0.0164093 ]]], dtype=float32)
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'
zs <xarray.DataArray (x: 2, y: 2, z: 2)> Size: 32B
array([[[0.90465474, 0.9038168 ],
        [0.89814764, 0.892997  ]],

       [[0.8973426 , 0.89534825],
        [0.90154576, 0.9031705 ]]], dtype=float32)
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'


In [100]:
density_torch, zn0_torch = get_density_at_surface_tensor(A_torch,B_torch,mask)

zt tensor([[[0.0087, 0.0225],
         [0.0207, 0.0167]],

        [[0.0134, 0.0013],
         [0.0193, 0.0164]]])
zs tensor([[[0.9047, 0.9038],
         [0.8981, 0.8930]],

        [[0.8973, 0.8953],
         [0.9015, 0.9032]]])


/var/folders/t8/ssf_3b9x74ldhyhx0tc_79000000gn/T/ipykernel_12275/329476930.py:91: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  rhop = zn0 * ztm  # potential density referenced at the surface


In [101]:
density_torch

tensor([[[   0.0000, 1000.5630],
         [1000.2266,  999.9176]],

        [[1000.1656, 1000.0234],
         [1000.4239, 1000.5143]]], dtype=torch.float64)

In [102]:
density

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 64B
array([[[   0.        , 1000.56774902],
        [1000.23132324,  999.92181396]],

       [[1000.16888428, 1000.02380371],
        [1000.42810059, 1000.51782227]]])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'

In [104]:
C_torch = torch.tensor(A.data, dtype=torch.float32, device='mps')
C_torch

tensor([[[0.3489, 0.8999],
         [0.8277, 0.6675]],

        [[0.5342, 0.0515],
         [0.7724, 0.6564]]], device='mps:0')

In [115]:
    rdeltaS = 32.0
    r1_S0 = 0.875 / 35.16504
    r1_T0 = 1.0 / 40.0
    r1_Z0 = 1.0e-4

    EOS000 = 8.0189615746e02
    EOS100 = 8.6672408165e02
    EOS200 = -1.7864682637e03
    EOS300 = 2.0375295546e03
    EOS400 = -1.2849161071e03
    EOS500 = 4.3227585684e02
    EOS600 = -6.0579916612e01
    EOS010 = 2.6010145068e01
    EOS110 = -6.5281885265e01
    EOS210 = 8.1770425108e01
    EOS310 = -5.6888046321e01
    EOS410 = 1.7681814114e01
    EOS510 = -1.9193502195
    EOS020 = -3.7074170417e01
    EOS120 = 6.1548258127e01
    EOS220 = -6.0362551501e01
    EOS320 = 2.9130021253e01
    EOS420 = -5.4723692739
    EOS030 = 2.1661789529e01
    EOS130 = -3.3449108469e01
    EOS230 = 1.9717078466e01
    EOS330 = -3.1742946532
    EOS040 = -8.3627885467
    EOS140 = 1.1311538584e01
    EOS240 = -5.3563304045
    EOS050 = 5.4048723791e-01
    EOS150 = 4.8169980163e-01
    EOS060 = -1.9083568888e-01
    EOS001 = 1.9681925209e01
    EOS101 = -4.2549998214e01
    EOS201 = 5.0774768218e01
    EOS301 = -3.0938076334e01
    EOS401 = 6.6051753097
    EOS011 = -1.3336301113e01
    EOS111 = -4.4870114575
    EOS211 = 5.0042598061
    EOS311 = -6.5399043664e-01
    EOS021 = 6.7080479603
    EOS121 = 3.5063081279
    EOS221 = -1.8795372996
    EOS031 = -2.4649669534
    EOS131 = -5.5077101279e-01
    EOS041 = 5.5927935970e-01
    EOS002 = 2.0660924175
    EOS102 = -4.9527603989
    EOS202 = 2.5019633244
    EOS012 = 2.0564311499
    EOS112 = -2.1311365518e-01
    EOS022 = -1.2419983026
    EOS003 = -2.3342758797e-02
    EOS103 = -1.8507636718e-02
    EOS013 = 3.7969820455e-01

In [116]:
A

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 64B
array([[[0.99866692, 0.79439934],
        [0.33171298, 0.28711696]],

       [[0.57617926, 0.9067433 ],
        [0.05857525, 0.38869932]]])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'

In [117]:
A_torch

tensor([[[0.9987, 0.7944],
         [0.3317, 0.2871]],

        [[0.5762, 0.9067],
         [0.0586, 0.3887]]])

In [124]:
temp = 10*A
sal = 33*A

In [125]:
zt = temp * r1_T0  # temperature
zt_torch = 10 * A_torch * r1_T0  # temperature
print(zt)
print(zt_torch)

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 64B
array([[[0.24966673, 0.19859984],
        [0.08292825, 0.07177924]],

       [[0.14404481, 0.22668582],
        [0.01464381, 0.09717483]]])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'
tensor([[[0.2497, 0.1986],
         [0.0829, 0.0718]],

        [[0.1440, 0.2267],
         [0.0146, 0.0972]]])


In [126]:
zs = np.sqrt(np.abs(sal + rdeltaS) * r1_S0)
zs_torch = torch.sqrt(torch.abs(33*A_torch + rdeltaS) * r1_S0)
print(zs)
print(zs_torch)

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 64B
array([[[1.27132952, 1.20355678],
        [1.03374281, 1.01587654]],

       [[1.1266598 , 1.24128873],
        [0.91888146, 1.05613317]]])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'
tensor([[[1.2713, 1.2036],
         [1.0337, 1.0159]],

        [[1.1267, 1.2413],
         [0.9189, 1.0561]]])


In [ ]:
    zn0 = (
        (
            (
                (
                    (
                        (EOS060 * zt + EOS150 * zs + EOS050) * zt
                        + (EOS240 * zs + EOS140) * zs
                        + EOS040
                    )
                    * zt
                    + ((EOS330 * zs + EOS230) * zs + EOS130) * zs
                    + EOS030
                )
                * zt
                + (((EOS420 * zs + EOS320) * zs + EOS220) * zs + EOS120) * zs
                + EOS020
            )
            * zt
            + ((((EOS510 * zs + EOS410) * zs + EOS310) * zs + EOS210) * zs + EOS110)
            * zs
            + EOS010
        )
        * zt
        + (
            ((((EOS600 * zs + EOS500) * zs + EOS400) * zs + EOS300) * zs + EOS200) * zs
            + EOS100
        )
        * zs
        + EOS000
    )

In [ ]:
zn0 = (((((((EOS060 * zt + EOS150 * zs + EOS050) * zt + (EOS240 * zs + EOS140) * zs + EOS040) * zt
              + ((EOS330 * zs + EOS230) * zs + EOS130) * zs + EOS030) * zt
              + (((EOS420 * zs + EOS320) * zs + EOS220) * zs + EOS120) * zs + EOS020) * zt
              + ((((EOS510 * zs + EOS410) * zs + EOS310) * zs + EOS210) * zs + EOS110) * zs + EOS010) * zt
              + ((((EOS600 * zs + EOS500) * zs + EOS400) * zs + EOS300) * zs + EOS200) * zs + EOS100) * zs + EOS000)

zn0_torch = ((((((EOS060 * zt_torch + EOS150 * zs_torch + EOS050) * zt_torch + (EOS240 * zs_torch + EOS140) * zs_torch + EOS040) * zt_torch
              + ((EOS330 * zs_torch + EOS230) * zs_torch + EOS130) * zs_torch + EOS030) * zt_torch
              + (((EOS420 * zs_torch + EOS320) * zs_torch + EOS220) * zs_torch + EOS120) * zs_torch + EOS020) * zt_torch
              + ((((EOS510 * zs_torch + EOS410) * zs_torch + EOS310) * zs_torch + EOS210) * zs_torch + EOS110) * zs_torch + EOS010) * zt_torch
              + (((((EOS600 * zs_torch + EOS500) * zs_torch + EOS400) * zs_torch + EOS300) * zs_torch + EOS200) * zs_torch + EOS100) * zs_torch + EOS000)

print(zn0)
print(zn0_torch)

<xarray.DataArray (x: 2, y: 2, z: 2)> Size: 64B
array([[[1024.9500991 , 1020.18662495],
        [1008.67881682, 1007.52017874]],

       [[1014.87865488, 1022.83157519],
        [1001.44374374, 1010.14706674]]])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
  * y        (y) int64 16B 10 20
  * z        (z) <U1 8B 'P' 'Q'
tensor([[[1025.2451, 1020.3113],
         [1008.6788, 1007.5198]],

        [[1014.9091, 1023.0392],
         [1001.4462, 1010.1490]]])


In [128]:
print(zn0.mean(axis=(1,2)))
print(zn0_torch.mean(dim=(1,2)))

<xarray.DataArray (x: 2)> Size: 16B
array([1015.3339299 , 1012.32526014])
Coordinates:
  * x        (x) <U1 8B 'A' 'B'
tensor([1015.3337, 1012.3252])


In [ ]:
((((((EOS060 * zt + EOS150 * zs + EOS050) * zt+ (EOS240 * zs + EOS140) * zs+ EOS040)
        * zt+ ((EOS330 * zs + EOS230) * zs + EOS130) * zs+ EOS030)
        * zt
        + (((EOS420 * zs + EOS320) * zs + EOS220) * zs + EOS120) * zs+ EOS020)
        * zt+ ((((EOS510 * zs + EOS410) * zs + EOS310) * zs + EOS210) * zs + EOS110)* zs+ EOS010)
        * zt+ (((((EOS600 * zs + EOS500) * zs + EOS400) * zs + EOS300) * zs + EOS200) * zs+ EOS100)
        * zs+ EOS000)

In [ ]:
((((((EOS060 * zt_torch + EOS150 * zs_torch + EOS050) * zt_torch + (EOS240 * zs_torch + EOS140) * zs_torch + EOS040) * zt_torch
              + ((EOS330 * zs_torch + EOS230) * zs_torch + EOS130) * zs_torch + EOS030) * zt_torch
              + (((EOS420 * zs_torch + EOS320) * zs_torch + EOS220) * zs_torch + EOS120) * zs_torch + EOS020) * zt_torch
              + ((((EOS510 * zs_torch + EOS410) * zs_torch + EOS310) * zs_torch + EOS210) * zs_torch + EOS110) * zs_torch + EOS010) * zt_torch
              + (((((EOS600 * zs_torch + EOS500) * zs_torch + EOS400) * zs_torch + EOS300) * zs_torch + EOS200) * zs_torch + EOS100) * zs_torch + EOS000)

In [5]:

ls

Analyse-inference-results.ipynb    easy-example.ipynb
Illustration.ipynb                 predictions_analysis.ipynb
Noise-data-schedule.ipynb          prepare_data.ipynb
Re-normalise-generated-data.ipynb  test_backbone.ipynb
dask-worker-space/                 test_notebook.ipynb
data-analysis.ipynb                visualise_generated.ipynb
data_loading.ipynb                 visualise_nc_fields.ipynb


In [18]:
import torch
import numpy as np
import torch.nn.functional as F
import sys
from configs.base_config import TrainingConfig
from utils import get_dataloader
import xarray as xr

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
batch = 8

# adding Folder_2 to the system path
sys.path.insert(0, '../analysis_scripts/')
sys.path.insert(0, '../../Diffusion_ Model')

from data_analytics import get_transformed_data
from metrics import get_density_at_surface_tensor

#mean density of the training set
mean_density = torch.tensor(np.loadtxt('../analysis_scripts/mean_density_train.txt'), device='mps', dtype=torch.float32)


config = TrainingConfig()
config.normalisation = '3-std'
config.data_file = '../../../DATA_DINOFusion/dino_1_4_degree_coarse_240125.tar'
train_dataloader = get_dataloader(config.data_file, batch_size=batch,
                                        fields=config.fields, normalisation='3-std', transform=True, shuffle=True, device=device)
extractor = train_dataloader.get_transform().uncall

file_mask_LR = xr.open_dataset("../analysis_scripts/data/DINO_1deg_mesh_mask_david_renamed.nc").sel(time_counter=0)
mask = file_mask_LR.rename({"nav_lev":"depth","y":"nav_lat","x":"nav_lon"})

/Users/blandinegorce/Documents/StageLOCEAN/code/.venv/lib/python3.12/site-packages/webdataset/compat.py:389: UserWarning: WebDataset(shardshuffle=...) is None; set explicitly to False or a number
  warnings.warn(


Reading infos in ../../../DATA_DINOFusion/dino_1_4_degree_coarse_240125.tar


In [34]:
density = mean_density.view(1, 36, 1, 1).expand(8, 36, 199, 62).clone()

In [35]:
dz = torch.tensor(mask.e3t_0.values, device=device, dtype=torch.float32)
grad_density = (density[:,:-1,:,:] - density[:,1:,:,:]) / dz[:-1,:,:]

tensor([], device='mps:0')

In [23]:

tmask = torch.tensor(mask.tmask.values, device=device, dtype=torch.float32).repeat(batch, 1, 1, 1)
tmask.shape

torch.Size([8, 36, 199, 62])

In [ ]:
x = mean_density.repeat(batch, 1, 1, 1)
    
for i in range(1):
#detach to remove gradients
    x = x.detach()

    with torch.enable_grad():
        x.requires_grad =True
        samples = get_transformed_data(x, function=extractor)
        
        #compute density
        tmask = torch.tensor(mask.tmask.values, device=device, dtype=torch.float32).repeat(batch, 1, 1, 1)
        density = get_density_at_surface_tensor(samples['toce'], samples['soce'], tmask)

        #remove nan 
        density[density != density] = 0

        #compute density vertical gradient
        dz = torch.tensor(mask.e3t_0.values, device=device, dtype=torch.float32)
        grad_density = (density[:,:-1,:,:] - density[:,1:,:,:]) / dz[:-1,:,:]

        loss = torch.sum(grad_density)

    loss.backward()

    grad = x.grad

    beta = self.beta.get_beta(t) if t is not None else self.beta
    print(f'apply constraint {beta}: {grad[0, :, 100,30]}')

    #x -= beta * self.relu(grad)
    x -= beta * grad

update = F.pad(grad_density, (1,1,5,4, 0,2), mode='constant', value=0)

self.beta.update(esp=torch.mean(update, dim=0), eta=0.0001, type='ineq')

